<a href="https://colab.research.google.com/github/BerkeleyExpertSystemTechnologiesLab/Squishy-Methane-Analysis/blob/jberry/Squish_Robot_Quant_Model_v5_1_mm_%2B_image_transforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Description

This model was produced by and for Squishy Robotics for the task of identifying and classifying methane leaks.


This model is for testing the efficiency on two parameters, PPM and Distance, using a small and basic Neural Network.


This model is experimental and uses the Optuna Hyperparameter Optimizer to search for successful hyperparameters (Learning Rate, Optimizer, Batch Size, Dropout %, etc...) and different optimizers. As such if you want to test a specific Model architecture you need to comment out the Optuna code and run a train/test on that specific model.

In [1]:
pip install optuna #Hyperparameter Optimizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 31.7 MB/s eta 0:00:00


In [2]:
import os
import numpy as np

from collections import defaultdict
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from torch.utils.data import random_split

# Hyperparameter Search
import optuna

import json
import glob

#For file uploading
from google.colab import files
from google.colab import drive
from google.colab import auth


In [3]:
#Upload the file
auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# This may take several minutes, the synthetic dataset can be large
!unzip -q "/content/drive/MyDrive/Squishy_Robotics_Dataset/Final_Dataset.zip" -d /content/

In [7]:
# Assuming the data is in 'Final_Dataset/data' and class folders are named 'class_0' ... 'class_7'
data_dir = 'Final_Dataset/normal/data'
classes = sorted(os.listdir(data_dir))
print(f"Classes: {classes}")

Classes: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7']


## Create a dataset and dataloader

In [8]:
class Metadata_Only_Dataset(Dataset):
    def __init__(self, json_files, labels):
        """
        numpy_dir points to all the numpy 2 channel frames that were collected
          from METEC. This is designed to be 1st Channel Greyscale image of
          background, 2nd channel is just the gas plume scaled to some ppm
        json_dir points to all the metadata (ppm, distance, etc) that was
          collected from METEC or estimated using BEST Labs algorithms
        """

        self.json_files = json_files
        self.labels = labels


    def __len__(self):
      return len(self.json_files)


    def __getitem__(self, idx):
      json_path = self.json_files[idx]
      with open(json_path, 'r') as f:
        metadata = json.load(f)

      metadat_features = self._extract_metadata_features(metadata)
      metadata_tensor = torch.tensor(metadat_features, dtype=torch.float32)

      label = self.labels[idx]

      return metadata_tensor, label


    def _extract_metadata_features(self, metadata):
      """
      Extracts a few entries from the metadata.
        For now:
          distance
          ppm
        In the future
          windspeed
          angle?
      """

      features = []

      # If the features exist, extract them, else place 0.0
      # Print warning statements if unable to retrieve the data
      distance = metadata.get("distance_m", None)
      if distance is None or distance == 0.0:
          print(f"WARNING: Invalid or missing distance_m value: {distance}")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(distance)

      ppm = metadata.get("ppm", None)
      if ppm is None:
          print(f"WARNING: Missing ppm value")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(ppm)

      return features

In [10]:
json_dir = "./Final_Dataset/normal/metadata"

all_json_files = []
all_labels = []

print(f"Looking in: {json_dir}")
print(f"Directory exists: {os.path.exists(json_dir)}\n")

for class_idx in range(8):
    json_class_dir = os.path.join(json_dir, f"class_{class_idx}")
    json_files_in_class = sorted(glob.glob(os.path.join(json_class_dir, "*.json")))

    print(f"Class {class_idx}: Found {len(json_files_in_class)} files")

    for json_file in json_files_in_class:
        all_json_files.append(json_file)
        all_labels.append(class_idx)

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_json_files)} json files")
print(f"{'='*60}\n")

# Video splitting - extract video_id from json filename
video_to_indices = defaultdict(list)
for idx, json_file in enumerate(all_json_files):
    video_id = os.path.basename(json_file).split('_')[0]
    video_to_indices[video_id].append(idx)

print(f"Number of unique videos: {len(video_to_indices)}")
print(f"Video IDs: {sorted(video_to_indices.keys())}\n")

Looking in: ./Final_Dataset/normal/metadata
Directory exists: True

Class 0: Found 28 files
Class 1: Found 28 files
Class 2: Found 28 files
Class 3: Found 28 files
Class 4: Found 28 files
Class 5: Found 28 files
Class 6: Found 28 files
Class 7: Found 28 files

TOTAL: 224 json files

Number of unique videos: 28
Video IDs: ['1237', '1238', '1239', '1240', '1241', '1242', '1467', '1468', '1469', '1470', '1471', '1472', '2559', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2569', '2571', '2578', '2579', '2580', '2581', '2583']



In [11]:
video_to_indices = defaultdict(list) #Make an empty dictionary of lists

for idx, json_file in enumerate(all_json_files):
    video_id = os.path.basename(json_file).split('_')[0] #Extract 4 digit code from numpy filename
    video_to_indices[video_id].append(idx)

video_ids = list(video_to_indices.keys())

# Split the video into train and test
train_vids, test_vids = train_test_split(video_ids, test_size=0.2, random_state=42)

# Verify no overlap
overlap = set(train_vids) & set(test_vids)
if overlap:
    print(f"\nVideos overlap: {overlap}")
else:
    print(f"\nNo video overlap - train and test are separate")

train_indices = []
test_indices = []

for vid in train_vids:
    train_indices.extend(video_to_indices[vid])
for vid in test_vids:
    test_indices.extend(video_to_indices[vid])

# Create file lists
train_json = [all_json_files[i] for i in train_indices]
train_labels_list = [all_labels[i] for i in train_indices]

test_json = [all_json_files[i] for i in test_indices]
test_labels_list = [all_labels[i] for i in test_indices]



No video overlap - train and test are separate


In [12]:
# SHOW FINAL SPLIT STATISTICS
print(f"\n{'='*60}")
print("DATASET STATISTICS")
print("="*90)

print(f"\nTRAINING SET:")
print(f"   Total samples: {len(train_json)}")
print(f"   From {len(train_vids)} videos: {sorted(train_vids)}")

# Count samples per class in training
train_class_counts = Counter(train_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = train_class_counts.get(class_id, 0)
    percentage = (count / len(train_json) * 100) if len(train_json) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

print(f"\nTEST SET:")
print(f"   Total samples: {len(test_json)}")
print(f"   From {len(test_vids)} videos: {sorted(test_vids)}")

# Count samples per class in testing
test_class_counts = Counter(test_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = test_class_counts.get(class_id, 0)
    percentage = (count / len(test_json) * 100) if len(test_json) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

# VERIFY ALL CLASSES PRESENT
print(f"\n{'='*70}")
print("VERIFICATION")
print("="*70)

train_classes = set(train_labels_list)
test_classes = set(test_labels_list)
missing_train = set(range(8)) - train_classes
missing_test = set(range(8)) - test_classes

if missing_train:
    print(f"WARNING: Training missing classes {missing_train}")
else:
    print(f"Training set has all 8 classes")

if missing_test:
    print(f"WARNING: Testing missing classes {missing_test}")
else:
    print(f"Test set has all 8 classes")

# Show train/test split ratio
total_samples = len(train_json) + len(test_json)
train_ratio = len(train_json) / total_samples * 100
test_ratio = len(test_json) / total_samples * 100
print(f"\nSplit ratio: {train_ratio:.1f}% train / {test_ratio:.1f}% test")

print(f"\n{'='*70}")
print("DATA SPLIT COMPLETE AND VERIFIED")
print("="*70)



DATASET STATISTICS

TRAINING SET:
   Total samples: 176
   From 22 videos: ['1238', '1239', '1240', '1241', '1242', '1467', '1468', '1471', '1472', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2571', '2578', '2579', '2581', '2583']

   Samples per class:
      Class 0:    22 samples (12.50%)
      Class 1:    22 samples (12.50%)
      Class 2:    22 samples (12.50%)
      Class 3:    22 samples (12.50%)
      Class 4:    22 samples (12.50%)
      Class 5:    22 samples (12.50%)
      Class 6:    22 samples (12.50%)
      Class 7:    22 samples (12.50%)

TEST SET:
   Total samples: 48
   From 6 videos: ['1237', '1469', '1470', '2559', '2569', '2580']

   Samples per class:
      Class 0:     6 samples (12.50%)
      Class 1:     6 samples (12.50%)
      Class 2:     6 samples (12.50%)
      Class 3:     6 samples (12.50%)
      Class 4:     6 samples (12.50%)
      Class 5:     6 samples (12.50%)
      Class 6:     6 samples (12.50%)
      Class 7:     6 samples (12

In [13]:
train_dataset = Metadata_Only_Dataset(train_json,
                                    train_labels_list)
test_dataset = Metadata_Only_Dataset(test_json,
                                   test_labels_list)

# Define the CNN model

## Define the Optuna Objective Function

This function will be called by Optuna for each trial. It will:
1. Suggest hyperparameters using the trial object.
2. Build and train the CNN model with the suggested hyperparameters.
3. Evaluate the model on a validation set
4. Return the metric to minimize (loss) or maximize (accuracy).

In [14]:
def objective(trial):

    #############################
    # All Hyperparameters Tested
    #############################
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'AdamW'])
    momentum = trial.suggest_float('momentum', 0.0, 0.99) if optimizer_name in ['SGD'] else 0.0
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.01)
    hidden_size = trial.suggest_int('hidden_size', 64, 256)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    num_epochs = trial.suggest_int('num_epochs', 5, 10)
    fc_drop_rate = trial.suggest_float('fc_drop_rate', 0.2, 0.6)

    #####################
    # Define the Model
    #####################
    class MetadataNet(nn.Module):
        def __init__(self, num_metadata_feats = 2, fc_drop_rate = 0.3):
            super(MetadataNet, self).__init__()

            self.metadata_fc1 = nn.Linear(num_metadata_feats, 64)
            self.metadata_bn1 = nn.BatchNorm1d(64)
            self.metadata_relu1 = nn.ReLU()
            self.metadata_dropout = nn.Dropout(fc_drop_rate)

            self.fc1 = nn.Linear(64, hidden_size)
            self.bn4 = nn.BatchNorm1d(hidden_size)
            self.relu4 = nn.ReLU()
            self.dropout4 = nn.Dropout(fc_drop_rate)
            self.fc2 = nn.Linear(hidden_size, 8)

        def forward(self, metadata):
            # Convolutional Blocks
            x = self.metadata_relu1(self.metadata_bn1(self.metadata_fc1(metadata)))
            x = self.metadata_dropout(x)

            x = self.relu4(self.bn4(self.fc1(x)))
            x = self.dropout4(x)
            output = self.fc2(x)


            return output

    model = MetadataNet(num_metadata_feats = 2, fc_drop_rate=fc_drop_rate)

    ###############################
    # Define optimizer
    ###############################
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'Adadelta':
        optimizer = optim.Adadelta(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "Muon":
        optimizer = optim.Muon(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer name: {optimizer_name}")

    criterion = nn.CrossEntropyLoss()

    ##########################################
    # Create DataLoaders with trial batch_size
    ##########################################
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    ###############################
    # Train the model
    ###############################
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)


    print(f"\n{'='*70}")
    print(f"Trial {trial.number} | lr={lr:.6f} | optimizer={optimizer_name} | "
          f"batch={batch_size} | hidden={hidden_size}")
    print(f"{'='*70}")


    model.train()
    train_correct = 0
    train_total = 0
    for epoch in range(num_epochs):
        train_correct = 0
        train_total = 0
        train_loss = 0.0
        num_batches = 0

        for metadata, labels in train_loader:
            metadata = metadata.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Calculate loss
            train_loss += loss.item()
            num_batches += 1

            # Calculate training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        #Print out training during each epoch
        train_accuracy = train_correct / train_total
        avg_train_loss = train_loss / num_batches

        # Print training accuracy for this epoch
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")

    #####################
    # Evaluate the model
    #####################
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    num_val_batches = 0
    with torch.no_grad():

        for metadata, labels in test_loader:
            metadata = metadata.to(device)
            labels = labels.to(device)

            # Calculate validation loss
            outputs = model(metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            num_val_batches += 1

            # Calculate Validation Accuract
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_val_loss = val_loss / num_val_batches

    print(f"Validation Loss: {avg_val_loss:.4f} | Validation Acc: {accuracy:.4f}")
    print(f"{'='*70}\n")

    return accuracy


## Run the Optuna Study

Now we will create an Optuna study and run the optimization process.

In [15]:
# Create a study object and specify the direction of optimization (maximize accuracy)
study = optuna.create_study(direction='maximize',
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))

# Run the optimization
study.optimize(objective, n_trials = 30)

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)

# Print the best accuracy found
print("Best accuracy: ", study.best_value)

# Plot the visualization
optuna.visualization.plot_param_importances(study).show()

# Run more trials
# study.optimize(objective, n_trials=20)

[I 2026-01-03 03:22:15,508] A new study created in memory with name: no-name-a3c17d36-6adb-49d7-84e0-d93ca6a45899



Trial 0 | lr=0.001804 | optimizer=SGD | batch=128 | hidden=165


[I 2026-01-03 03:22:16,756] Trial 0 finished with value: 0.125 and parameters: {'lr': 0.0018043954845701768, 'optimizer': 'SGD', 'momentum': 0.5749809954757253, 'weight_decay': 0.0012015222013029715, 'hidden_size': 165, 'batch_size': 128, 'num_epochs': 9, 'fc_drop_rate': 0.428633271982659}. Best is trial 0 with value: 0.125.


Epoch [ 1/9] Train Loss: 2.1393 | Train Acc: 0.1307
Epoch [ 2/9] Train Loss: 2.1833 | Train Acc: 0.1307
Epoch [ 3/9] Train Loss: 2.1978 | Train Acc: 0.1420
Epoch [ 4/9] Train Loss: 2.1623 | Train Acc: 0.1364
Epoch [ 5/9] Train Loss: 2.1788 | Train Acc: 0.1420
Epoch [ 6/9] Train Loss: 2.2131 | Train Acc: 0.1023
Epoch [ 7/9] Train Loss: 2.2171 | Train Acc: 0.1080
Epoch [ 8/9] Train Loss: 2.1703 | Train Acc: 0.1080
Epoch [ 9/9] Train Loss: 2.2088 | Train Acc: 0.1193
Validation Loss: 2.0825 | Validation Acc: 0.1250


Trial 1 | lr=0.000046 | optimizer=Adam | batch=64 | hidden=140


[I 2026-01-03 03:22:16,978] Trial 1 finished with value: 0.125 and parameters: {'lr': 4.614799774499509e-05, 'optimizer': 'Adam', 'weight_decay': 0.006595354541919543, 'hidden_size': 140, 'batch_size': 64, 'num_epochs': 8, 'fc_drop_rate': 0.2791472891660492}. Best is trial 0 with value: 0.125.


...

Epoch [ 2/6] Train Loss: 2.0229 | Train Acc: 0.2102
Epoch [ 3/6] Train Loss: 1.9321 | Train Acc: 0.2273
Epoch [ 4/6] Train Loss: 1.8465 | Train Acc: 0.3068
Epoch [ 5/6] Train Loss: 1.8915 | Train Acc: 0.2216
Epoch [ 6/6] Train Loss: 1.7564 | Train Acc: 0.2955
Validation Loss: 1.3123 | Validation Acc: 0.5833


Trial 27 | lr=0.002239 | optimizer=Adam | batch=128 | hidden=188
Epoch [ 1/7] Train Loss: 2.1940 | Train Acc: 0.0909
Epoch [ 2/7] Train Loss: 2.0156 | Train Acc: 0.1591
Epoch [ 3/7] Train Loss: 1.9441 | Train Acc: 0.2386
Epoch [ 4/7] Train Loss: 1.8955 | Train Acc: 0.2216


[I 2026-01-03 03:22:21,632] Trial 27 finished with value: 0.3333333333333333 and parameters: {'lr': 0.002239103307781731, 'optimizer': 'Adam', 'weight_decay': 0.009806272989273015, 'hidden_size': 188, 'batch_size': 128, 'num_epochs': 7, 'fc_drop_rate': 0.2312708593137729}. Best is trial 17 with value: 0.625.
[I 2026-01-03 03:22:21,741] Trial 28 finished with value: 0.2916666666666667 and parameters: {'lr': 0.014486029861409795, 'optimizer': 'AdamW', 'weight_decay': 0.00854534671208614, 'hidden_size': 164, 'batch_size': 64, 'num_epochs': 6, 'fc_drop_rate': 0.3320108101966525}. Best is trial 17 with value: 0.625.


Epoch [ 5/7] Train Loss: 1.8773 | Train Acc: 0.2557
Epoch [ 6/7] Train Loss: 1.8149 | Train Acc: 0.2727
Epoch [ 7/7] Train Loss: 1.7759 | Train Acc: 0.2784
Validation Loss: 1.7513 | Validation Acc: 0.3333


Trial 28 | lr=0.014486 | optimizer=AdamW | batch=64 | hidden=164
Epoch [ 1/6] Train Loss: 2.1360 | Train Acc: 0.1648
Epoch [ 2/6] Train Loss: 1.8335 | Train Acc: 0.2443
Epoch [ 3/6] Train Loss: 1.7327 | Train Acc: 0.2557
Epoch [ 4/6] Train Loss: 1.7513 | Train Acc: 0.2500
Epoch [ 5/6] Train Loss: 1.6264 | Train Acc: 0.2898
Epoch [ 6/6] Train Loss: 1.5835 | Train Acc: 0.3295
Validation Loss: 1.3406 | Validation Acc: 0.2917


Trial 29 | lr=0.003762 | optimizer=Adam | batch=128 | hidden=213
Epoch [ 1/8] Train Loss: 2.1457 | Train Acc: 0.1420
Epoch [ 2/8] Train Loss: 1.9376 | Train Acc: 0.1932
Epoch [ 3/8] Train Loss: 2.0639 | Train Acc: 0.1875


[I 2026-01-03 03:22:21,859] Trial 29 finished with value: 0.3333333333333333 and parameters: {'lr': 0.0037620806472334666, 'optimizer': 'Adam', 'weight_decay': 0.004461581819783113, 'hidden_size': 213, 'batch_size': 128, 'num_epochs': 8, 'fc_drop_rate': 0.4508400994667349}. Best is trial 17 with value: 0.625.


Epoch [ 4/8] Train Loss: 1.9211 | Train Acc: 0.2670
Epoch [ 5/8] Train Loss: 1.9773 | Train Acc: 0.1875
Epoch [ 6/8] Train Loss: 1.8798 | Train Acc: 0.2955
Epoch [ 7/8] Train Loss: 1.9286 | Train Acc: 0.2386
Epoch [ 8/8] Train Loss: 1.8477 | Train Acc: 0.2159
Validation Loss: 1.7171 | Validation Acc: 0.3333

Best hyperparameters:  {'lr': 0.01238067381929555, 'optimizer': 'Adam', 'weight_decay': 0.008511287031819364, 'hidden_size': 164, 'batch_size': 16, 'num_epochs': 5, 'fc_drop_rate': 0.26739084879754105}
Best accuracy:  0.625


#Sources:
###Hyperparameter Tuning with Optuna:
https://medium.com/@taeefnajib/hyperparameter-tuning-using-optuna-c46d7b29a3e

https://optuna.org/#code_examples
###Multi-Modal ML Models
https://www.nature.com/articles/s41598-025-14901-4
https://www.reddit.com/r/MachineLearning/comments/nziumg/combining_images_and_other_numeric_features_in_a/
https://pyimagesearch.com/2019/02/04/keras-multiple-inputs-and-mixed-data/

###Next Models to test:
VideoGasNet:
https://www.sciencedirect.com/science/article/pii/S0360544221017643

GasVit: https://www.sciencedirect.com/science/article/pii/S1568494623011560?via%3Dihub#sec3